[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/cours/seance3_cours.ipynb)

# Séance 2.3 — Agréger, croiser et visualiser

**Cours** · durée : 2h — sept techniques, chacune suivie d'exercices

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- répondre à « combien par ... ? » avec `groupby`
- calculer plusieurs indicateurs d'un coup avec `agg`
- rassembler trois fichiers en une seule table avec `merge`
- croiser deux dimensions dans un tableau de synthèse
- choisir le bon graphique selon la question posée
- produire une courbe, des barres, un histogramme et un nuage de points
- rendre un graphique lisible : titre, axes, unités
- repérer ce qu'un graphique cache autant que ce qu'il montre
- mettre un chiffre sur un lien entre deux grandeurs avec `corr()`

## Retour à la question de départ

> *« Sur quel marché faut-il investir l'an prochain ? »*

Vous savez maintenant charger et nettoyer. Et pourtant vous ne pouvez toujours
pas répondre — pour une raison très simple :

**`ventes.csv` ne contient pas le pays.** Il contient un `client_id`. Le pays
est dans `clients.csv`.

C'est la situation normale en entreprise : l'information est **répartie entre
plusieurs fichiers**, et la réponse naît de leur croisement. C'est l'objet de
cette séance.

La séance se lit en deux temps : d'abord **répondre** à la question, avec
`groupby`, `merge` et les tableaux croisés ; puis la **donner à voir**, avec
quatre graphiques.

> 📋 **Comment on travaille aujourd'hui.** Sept techniques. Pour chacune : une
> démonstration que vous suivez, puis **un ou deux exercices que vous faites**
> — les uns ont des `____` à remplir, les autres sont une cellule vide où vous
> écrivez tout. La cellule de vérification vous dit immédiatement si votre
> réponse est bonne.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")       ## une ligne = un produit
clients = pd.read_csv(BASE + "clients.csv")     ## une ligne = un client
produits = pd.read_csv(BASE + "produits.csv")   ## une ligne = une reference

ventes["ca"] = ventes["qte"] * ventes["prix"]   ## le CA de chaque ligne
ventes["date"] = pd.to_datetime(ventes["date"]) ## du texte vers des dates
print(ventes.shape, clients.shape, produits.shape)

## 1. `groupby` — la commande la plus utile de tout le bloc

`groupby` répond à toutes les questions de la forme **« combien par ... ? »**.

Trois temps, toujours les mêmes :

1. **Découper** les lignes en paquets selon une colonne
2. **Calculer** un indicateur dans chaque paquet
3. **Recoller** les résultats en un tableau

In [ ]:
# "Combien de chiffre d'affaires par client ?"
ca_client = ventes.groupby("client_id")["ca"].sum()   ## decouper, calculer, recoller

ca_client.nlargest(5).round(2)   ## les cinq plus gros clients

*Nouveau :* `serie.nlargest(5)` garde les **cinq plus grandes valeurs**,
déjà classées. Sur un tableau, on précise la colonne : `df.nlargest(5, "ca")`.

Un client pèse à lui seul **143 825 €** — sur les 472 que compte le fichier.
Retenez le chiffre : la concentration est un risque, et vous allez la mesurer
tout à l'heure.

### Plusieurs indicateurs d'un coup — `agg`

In [ ]:
resume = ventes.groupby("client_id").agg(
    ca=("ca", "sum"),              # total depense
    nb_lignes=("cmd_id", "count"), # nombre de LIGNES
    nb_cmd=("cmd_id", "nunique"),  # nombre de COMMANDES distinctes
)
resume.nlargest(3, "ca").round(2)

La syntaxe se lit : `nom_voulu=("colonne_source", "operation")`.

> ⚠️ **`count` vs `nunique` — l'erreur classique.**
> `count` compte les **lignes**. `nunique` compte les **valeurs distinctes**.
> Une commande de 30 articles occupe 30 lignes mais reste **une** commande.
> Regardez l'écart entre `nb_lignes` et `nb_cmd` ci-dessus : confondre les
> deux, c'est diviser son panier moyen par 20.

Les opérations disponibles : `"sum"`, `"mean"`, `"median"`, `"min"`, `"max"`,
`"count"`, `"nunique"`, `"std"`.

---

### ✏️ Exercice — Le panier moyen

> **Votre mission :**
> - Un **panier**, c'est une commande entière — pas une ligne.
> - Calculer le CA total de chaque commande → `par_cmd`, puis le nombre de commandes → `nb_cmd_total` et le panier moyen arrondi à 2 décimales → `panier_moyen`.

In [ ]:
par_cmd = ventes.groupby("____")["ca"].sum()

nb_cmd_total = len(par_cmd)
panier_moyen = round(par_cmd.____(), 2)

print(nb_cmd_total, "commandes | panier moyen :", panier_moyen)

In [ ]:
verifier("nombre de commandes", nb_cmd_total == 1955,
         "groupby('cmd_id') : une ligne du resultat = une commande")
verifier("panier moyen", panier_moyen == 589.73,
         "mean() sur le CA par commande, pas sur le CA par ligne")

---

### ✏️ Exercice — Le meilleur client, vraiment ?

> **Votre mission :**
> - Construire `profil` : une ligne par client, avec son CA total (`ca`) et son nombre de **commandes distinctes** (`nb_cmd`).
> - Ajouter une colonne `panier` = CA ÷ nombre de commandes.
> - Afficher le client au plus gros panier moyen. Puis le meilleur **parmi ceux qui ont au moins 5 commandes** → mettre son identifiant dans `client_fidele`.
> - La réponse change. Lequel des deux présenteriez-vous à un directeur commercial ?

In [ ]:
verifier("client fidele au meilleur panier", client_fidele == 12753,
         "filtrez sur nb_cmd >= 5 AVANT de classer par panier")

## 2. `merge` — rassembler les fichiers

C'est l'équivalent du `RECHERCHEV` d'Excel, en beaucoup plus sûr.

Les deux tables ont une colonne en commun : `client_id`. `merge` s'en sert
pour aller chercher, pour chaque vente, les informations du client
correspondant.

In [ ]:
avant = len(ventes)
vc = ventes.merge(clients, on="client_id")   ## on = la colonne commune

# LE reflexe : verifier qu'on n'a ni perdu ni duplique de lignes
print(avant, "->", len(vc))

> ⚠️ **Ne sautez jamais cette vérification.** Si la clé de jointure n'est pas
> unique dans la table de droite, `merge` **duplique** des lignes sans rien
> dire. Vos totaux deviennent faux et rien ne vous alerte. Deux nombres
> affichés, une seconde de lecture, et vous êtes tranquille.

`vc` contient maintenant les colonnes des deux tables :

In [ ]:
# pays et segment viennent de clients, qte et prix de ventes
vc[["client_id", "pays", "segment", "qte", "prix", "ca"]].head(3)

### Et enfin, la réponse à la question du bloc

In [ ]:
ca_pays = vc.groupby("pays")["ca"].sum().sort_values(ascending=False)

ca_pays.head(5).round(2)   ## le classement des marches

*Nouveau :* `sort_values(ascending=False)` classe du plus grand au plus
petit. Sans `ascending=False`, le classement part du plus petit.

Le Royaume-Uni domine — c'est le marché domestique, sans surprise.

**Mais regardez l'Irlande : 261 205 €, deuxième marché du groupe.** Combien
de clients irlandais y a-t-il ?

In [ ]:
vc.query("pays == 'Irlande'")["client_id"].nunique()   ## combien ?

**Deux.** Deux clients suffisent à faire le deuxième marché du groupe. Vous
allez chiffrer ce que ça représente dans l'exercice « Chiffrer l'anomalie
irlandaise », un peu plus bas.

---

### ✏️ Exercice — Le panier moyen par pays

> **Votre mission :**
> - À partir de `vc` : pour chaque pays, le CA total (`ca`) et le nombre de **commandes distinctes** (`nb_cmd`).
> - Ajouter une colonne `panier` = CA ÷ nombre de commandes, arrondie à 2 décimales.
> - Mettre le panier moyen irlandais dans `panier_irl`.

In [ ]:
parpays = vc.groupby("pays").agg(
    ca=("ca", "sum"),
    nb_cmd=("cmd_id", "____"),
)
parpays["panier"] = (parpays["ca"] / parpays["____"]).round(2)

panier_irl = parpays.loc["Irlande", "panier"]
print(panier_irl)

In [ ]:
verifier("panier moyen irlandais", panier_irl == 1020.33,
         "avec count au lieu de nunique le panier serait ridiculement bas")

---

### ✏️ Exercice — Chiffrer l'anomalie irlandaise

> **Votre mission :**
> - Quelle **part du chiffre d'affaires total** l'Irlande représente-t-elle, en % arrondi à 1 décimale ? → `part_irl`
> - Combien de clients irlandais y a-t-il ? → `nb_cli_irl`
> - Vous disposez déjà de `ca_pays` et de `vc`. Regardez les deux chiffres ensemble : que diriez-vous à un dirigeant ?

In [ ]:
verifier("part de l'Irlande", part_irl == 22.7,
         "divisez le CA irlandais par ca_pays.sum()")
verifier("clients irlandais", nb_cli_irl == 2,
         "nunique() sur client_id apres avoir filtre sur l'Irlande")

### La troisième table : les libellés des produits

Il reste `produits.csv`, qui donne le **libellé** et la **catégorie** de chaque
référence. Une deuxième jointure, sur `prod_id` cette fois, et les trois
fichiers n'en font plus qu'un.

In [ ]:
complet = vc.merge(produits, on="prod_id")   ## la 2e cle : prod_id

print(len(vc), "->", len(complet))   ## le meme reflexe qu'au-dessus
complet[["pays", "categorie", "qte", "prix", "ca"]].head(3)

**45 123 lignes des deux côtés : aucune n'a été perdue en chemin.**

`complet` porte maintenant tout : la date, le pays, le segment, la catégorie
et le chiffre d'affaires. C'est la table sur laquelle on travaillera pendant
toute la seconde moitié de la séance.

## 3. Croiser deux dimensions — le tableau de synthèse

`groupby` répond aux questions en **une** dimension : « combien par pays ? ».
Un dirigeant en pose presque toujours deux : « combien par pays **et** par
segment de clientèle ? »

C'est ce que fait un **tableau croisé** : une dimension en lignes, l'autre en
colonnes, et un chiffre dans chaque case.

In [ ]:
# le premier argument fait les lignes, le second les colonnes
pd.crosstab(vc["pays"], vc["segment"]).head(6)

Chaque case compte les **lignes** qui tombent dans ce croisement : 4 414 lignes
de vente allemandes viennent de clients « premium ».

Compter, c'est bien. **Additionner une grandeur**, c'est mieux — deux arguments
de plus suffisent : `values=` dit quelle colonne additionner, `aggfunc=` dit
comment.

In [ ]:
pd.crosstab(vc["pays"], vc["segment"],
            values=vc["ca"], aggfunc="sum").round(0).head(6)

Les mêmes cases, en euros cette fois. Et le Canada affiche des cases vides :
son unique client est « standard », il n'y a **aucune** vente canadienne
premium à additionner.

> ⚠️ **Une case vide dans un tableau croisé est une information, pas un bug.**
> Elle dit qu'un croisement n'existe pas dans vos données.

---

### ✏️ Exercice — Le tableau croisé complet

> **Votre mission :**
> - Croiser `pays` (en **lignes**) et `segment` (en **colonnes**) sur tous les pays, avec la somme du `ca` → `tableau`.
> - Mettre le CA des clients « premium » français dans `fr_premium`, arrondi à 0 décimale.
> - *Rappel :* `tableau.loc["France", "premium"]` va chercher une case par son nom de ligne, puis son nom de colonne.

In [ ]:
tableau = pd.crosstab(vc["____"], vc["____"],
                      values=vc["ca"], aggfunc="sum")

fr_premium = round(tableau.loc["France", "premium"], 0)
print(fr_premium)
tableau.head(4).round(0)

In [ ]:
verifier("premium francais", fr_premium == 114433.0,
         "pays en premier (les lignes), segment en second (les colonnes)")

---

## Et maintenant : donner à voir

Vous avez la réponse. Il reste à la **faire voir** — à vous d'abord, à un
dirigeant ensuite.

Un tableau de treize lignes de chiffre d'affaires mensuel se lit. Mais l'avez-
vous **vu** ? La même chose en courbe, et la tendance saute aux yeux en une
seconde.

> **Un graphique ne décore pas un rapport : il fait voir ce qu'un tableau
> cache.** Corollaire souvent oublié — si un tableau de trois lignes suffit,
> ne faites pas de graphique.

### Choisir le bon graphique

| Votre question | Le graphique |
|---|---|
| Comment ça évolue **dans le temps** ? | une **courbe** |
| Qui est le plus gros ? Comment ça se **compare** ? | des **barres** |
| Comment les valeurs sont-elles **réparties** ? | un **histogramme** |
| Y a-t-il un **lien** entre deux grandeurs ? | un **nuage de points** |

Quatre questions, quatre graphiques. C'est presque tout ce dont vous aurez
besoin.

## 4. La courbe — l'évolution dans le temps

Une courbe montre **comment une grandeur bouge au fil du temps** : une saison
forte, une tendance qui monte, un décrochage. On la choisit dès que l'axe
horizontal est une date — et seulement dans ce cas.

In [ ]:
# to_period("M") regroupe toutes les dates d'un meme mois
ca_mois = ventes.groupby(ventes["date"].dt.to_period("M"))["ca"].sum()
ca_mois.index = ca_mois.index.astype(str)   ## en texte : matplotlib prefere

ca_mois.plot(kind="line", marker="o", figsize=(7, 4))   ## une evolution
plt.title("Chiffre d'affaires mensuel")   ## sans titre, ce n'est pas un livrable
plt.ylabel("CA (euros)")                  ## la grandeur ET son unite
plt.xticks(rotation=45)                   ## des dates inclinees se lisent
plt.show()

Une montée régulière jusqu'à un pic en octobre, puis une chute brutale en
décembre.

**Que concluez-vous ?** Prenez trente secondes avant de continuer.

In [ ]:
# Verifions quelque chose avant de conclure
decembre = ventes.query("date >= '2011-12-01'")

print("derniere date du fichier :", ventes["date"].max().date())   ## le 9 !
print("jours de decembre 2011 presents :", decembre["date"].dt.day.nunique())

> ⚠️ **Il n'y a pas eu d'effondrement en décembre.** Le fichier s'arrête au
> **9 décembre**. On compare 8 jours de vente à des mois complets de 30 jours.
>
> Le graphique ne mentait pas. C'est la lecture qui était fausse.

C'est **l'erreur d'analyse la plus fréquente en entreprise**, et l'une des
plus coûteuses. Avant d'interpréter une évolution, vérifiez toujours que
**toutes les périodes sont comparables**.

Le vrai pic, lui, est bien réel : octobre. Pour un grossiste, c'est logique —
les détaillants se réapprovisionnent **avant** Noël, pas pendant.

---

### ✏️ Exercice — La courbe, avec titre et unité

> **Votre mission :**
> - Tracer le **nombre de commandes distinctes** par mois sous forme de courbe.
> - Titre et unité obligatoires : un graphique sans légende n'est pas un graphique, c'est un dessin.
> - Incliner les étiquettes à 45° et appeler `tight_layout()` : sur un petit écran, sans ça les dates se chevauchent ou sont coupées.
> - Mettre le mois qui compte le plus de commandes dans `mois_cmd`, et le nombre de mois du fichier dans `nb_mois`.

In [ ]:
nb_cmd = complet.groupby(complet["date"].dt.to_period("M"))["cmd_id"].____()
nb_cmd.index = nb_cmd.index.astype(str)

nb_cmd.plot(kind="____", marker="o", figsize=(7, 4))
plt.title("Nombre de commandes par mois")
plt.ylabel("____")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

mois_cmd = nb_cmd.idxmax()
nb_mois = len(nb_cmd)
print(mois_cmd, "|", nb_mois, "mois")

In [ ]:
verifier("mois record en commandes", mois_cmd == "2011-11",
         "nunique() compte les commandes distinctes, count() compterait les lignes")
verifier("nombre de mois", nb_mois == 13,
         "decembre 2010 et decembre 2011 comptent tous les deux")

---

### ✏️ Exercice — Deux marchés sur la même figure

> **Votre mission :**
> - Comparer l'évolution mensuelle de la France et de l'Allemagne **sur un seul graphique**.
> - Mettre le meilleur mois français dans `mois_fr`.
> - Deux courbes sur une figure se comparent ; deux figures côte à côte, non.

In [ ]:
deux = complet.query("pays in ['France', '____']").copy()
deux["mois"] = deux["date"].dt.to_period("M").astype(str)

empile = deux.groupby(["mois", "pays"])["ca"].sum()   ## deux cles de regroupement
par_mois = empile.____()                              ## les pays passent en colonnes
print(empile.shape, "->", par_mois.shape)

par_mois.plot(kind="line", marker="o", figsize=(7, 4))
plt.title("France et Allemagne, mois par mois")
plt.ylabel("CA (euros)")
plt.show()

mois_fr = par_mois["France"].idxmax()
print(mois_fr)

In [ ]:
verifier("meilleur mois francais", mois_fr == "2011-10",
         "empile.unstack() met les pays en colonnes ; idxmax() donne l'etiquette")

Quatre étapes nommées, et rien de magique :

1. `.copy()` puis une **colonne `mois`** : `dt.to_period("M")` réduit une date
   au mois qui la contient, `astype(str)` en fait du texte lisible en abscisse.
2. `groupby(["mois", "pays"])` — **deux clés** entre crochets au lieu d'une :
   un paquet par couple (mois, pays).
3. `unstack()` — le point à retenir.
4. `idxmax()` sur une colonne, comme au cycle précédent.

**Ce que `unstack()` fait à la forme du tableau**, et c'est tout ce qu'il fait :

| | avant `unstack()` | après |
|---|---|---|
| forme | **26 lignes**, une valeur | **13 lignes × 2 colonnes** |
| une ligne | un couple (mois, pays) | un mois |
| une colonne | — | un pays |

Et c'est ce qui rend le graphique possible : `plot` dessine **une courbe par
colonne**. Deux colonnes, deux courbes, et la légende arrive toute seule.

## 5. Les barres — comparer

Des barres **comparent des catégories entre elles** : quel pays pèse le plus,
quelle catégorie de produit arrive en tête. La longueur se lit d'un coup
d'œil, et l'ordre des barres fait la moitié du travail.

C'est le graphique du classement — celui qu'on met dans une note de synthèse
quand la question est « qui est le plus gros ? ».

**Toujours trier avant de tracer.** Un diagramme en barres non trié est
illisible.

In [ ]:
# nlargest(8) : le top 8. sort_values() : matplotlib dessine de bas en haut
ca_pays = complet.groupby("pays")["ca"].sum().nlargest(8).sort_values()

ca_pays.plot(kind="barh", figsize=(7, 4))   ## barh : les noms se lisent
plt.title("Chiffre d'affaires par pays (top 8)")
plt.xlabel("CA (euros)")
plt.show()

> 💡 **`barh` plutôt que `bar`.** En barres horizontales, les noms se lisent
> sans se chevaucher et sans rotation. Sur un écran étroit, c'est décisif.
>
> Et `sort_values()` **sans** `ascending=False` : matplotlib dessine de bas en
> haut, donc trier en ordre croissant met le plus grand tout en haut.

---

### ✏️ Exercice — Le rythme de la semaine

> **Votre mission :**
> - Calculer le CA par **jour de la semaine** dans `ca_jour`, trié du plus petit au plus grand.
> - Le tracer en barres horizontales, avec titre et unité.
> - Mettre le jour le plus fort dans `jour_top`.

In [ ]:
ca_jour = complet.groupby(complet["date"].dt.day_name())["ca"].sum().____()

ca_jour.plot(kind="____", figsize=(7, 4))
plt.title("Chiffre d'affaires par jour de la semaine")
plt.xlabel("CA (euros)")
plt.show()

jour_top = ca_jour.index[-1]
print(jour_top)

In [ ]:
verifier("jour le plus fort", jour_top == "Thursday",
         "sort_values() trie ; apres un tri croissant le plus grand est en position -1")

---

### ✏️ Exercice — Compter n'est pas sommer

> **Votre mission :**
> - Combien de **références différentes** chaque catégorie contient-elle ? → `nb_ref`, trié, en barres horizontales.
> - Mettre la catégorie la plus fournie dans `cat_ref`.
> - Attention : on compte des produits, on n'additionne pas des euros. Ce n'est pas le même graphique ni la même conclusion.

In [ ]:
nb_ref = produits.groupby("categorie").____().sort_values()

nb_ref.plot(kind="barh", figsize=(7, 4))
plt.title("Nombre de references par categorie")
plt.xlabel("references")
plt.show()

cat_ref = nb_ref.index[-1]
print(cat_ref)

In [ ]:
verifier("categorie la plus fournie", cat_ref == "deco",
         "size() compte les lignes, sum() additionnerait des valeurs")

## 6. L'histogramme — la répartition

Un histogramme montre **comment les valeurs d'une seule colonne se
répartissent** : où se concentre le gros de l'effectif, et jusqu'où traînent
les valeurs extrêmes. Il découpe la colonne en tranches et compte combien de
lignes tombent dans chacune.

On y vient quand une moyenne ne suffit plus — elle donne un chiffre, il donne
la **forme**.

In [ ]:
ventes["prix"].plot(kind="hist", bins=50, figsize=(7, 4))   ## 50 classes
plt.title("Repartition des prix unitaires")
plt.xlabel("Prix (euros)")
plt.show()

Illisible : une seule barre collée à gauche. En cause, le prix maximum à
4 161 €, qui étire tout l'axe.

**C'est une information, pas un problème.** Elle confirme ce qu'on avait vu
en séance 2.1 (moyenne 3,93 € contre médiane 1,95 €). Zoomons sur la zone
utile :

In [ ]:
# 85,8 % des ventes sont a moins de 5 euros
ventes.query("prix < 10")["prix"].plot(kind="hist", bins=40, figsize=(7, 4))

# Le zoom se dit DANS le titre : sinon on cache une information au lecteur
plt.title("Repartition des prix unitaires (moins de 10 euros)")
plt.xlabel("Prix (euros)")
plt.show()

Voilà l'entreprise réelle : **un vendeur de petits articles à moins de 5 €**,
en gros volumes. Ce n'est pas ce qu'une moyenne de 3,93 € laissait deviner —
elle aurait pu décrire aussi bien un catalogue homogène autour de 4 €.

> Quand vous zoomez pour rendre un graphique lisible, **dites-le dans le
> titre**. « moins de 10 euros » dans le titre ci-dessus : sans cette
> mention, vous cachez une information à votre lecteur.

---

### ✏️ Exercice — La répartition des quantités

> **Votre mission :**
> - Même technique, autre colonne : tracer l'histogramme des **quantités** commandées, en 50 classes.
> - Puis compter les lignes de plus de 100 unités → `nb_grosses`.

In [ ]:
complet["qte"].plot(kind="____", bins=50, figsize=(7, 4))
plt.title("Repartition des quantites commandees")
plt.xlabel("Quantite")
plt.show()

nb_grosses = len(complet.query("qte > ____"))
print(nb_grosses, "lignes de plus de 100 unites")

In [ ]:
verifier("lignes de plus de 100 unites", nb_grosses == 560,
         "le type de graphique est hist ; le seuil de la question est 100")

---

### ✏️ Exercice — Zoomer, et le dire

> **Votre mission :**
> - L'histogramme que vous venez de tracer est illisible : la commande à 1 440 unités étire tout l'axe.
> - Le retracer sur les seules lignes de **moins de 25 unités** → `petites`, et compter combien de lignes cela représente → `nb_petites`.
> - ⚠️ Le zoom doit apparaître **dans le titre** : sans ça, vous cachez une information à votre lecteur.

In [ ]:
petites = complet.query("qte < ____")
nb_petites = len(petites)

petites["qte"].plot(kind="hist", bins=25, figsize=(7, 4))
plt.title("Repartition des quantites (____)")   ## le zoom se dit ICI
plt.xlabel("Quantite")
plt.show()

print(nb_petites, "lignes sur", len(complet))

In [ ]:
verifier("lignes de moins de 25 unites", nb_petites == 41033,
         "filtrez avec query avant de tracer, puis len() sur le resultat")

## 7. Le nuage de points — chercher un lien

Un nuage de points croise **deux grandeurs chiffrées** : un point par ligne du
fichier, une grandeur en abscisse, l'autre en ordonnée. On y cherche une
forme — les points descendent-ils quand on va vers la droite ?

C'est le graphique de la question « ces deux choses vont-elles ensemble ? ».

In [ ]:
# random_state=42 : le meme echantillon a chaque execution
echantillon = ventes.query("prix < 20 and qte < 200").sample(2000, random_state=42)

# alpha=0.3 : des points translucides, sinon 2 000 points font une tache
echantillon.plot(kind="scatter", x="prix", y="qte", alpha=0.3, figsize=(7, 4))
plt.title("Quantite commandee selon le prix unitaire")
plt.xlabel("Prix unitaire (euros)")
plt.ylabel("Quantite")
plt.show()

Le nuage penche : les points les plus hauts sont à gauche. **Plus le prix
unitaire est élevé, plus les quantités commandées semblent faibles.** Attendu
— mais « semble » n'est pas « est », et un œil se laisse convaincre par pas
grand-chose.

> `alpha=0.3` rend les points semi-transparents : là où ils se superposent,
> la couleur devient plus dense. Sans ça, 2 000 points forment une bouillie
> noire.

> ⚠️ **Une relation n'est pas une cause.** Le prix ne « fait » pas baisser
> les quantités : ce sont deux conséquences du type de produit. Nous
> reviendrons sur cette distinction au bloc 5 (A/B testing) — c'est tout
> l'objet de l'expérimentation.

### Mesurer ce que l'œil croit voir — la corrélation

L'œil se trompe : une pente peut sauter aux yeux sur un échantillon et
disparaître sur le fichier entier. `.corr()` met un chiffre sur la relation.

In [ ]:
lien_echantillon = round(echantillon["prix"].corr(echantillon["qte"]), 3)

print(lien_echantillon)

**Un seul nombre, toujours entre −1 et +1.** Proche de **+1**, les deux
grandeurs montent ensemble ; proche de **−1**, l'une monte quand l'autre
descend ; proche de **0**, il n'y a pas de lien d'ensemble. Ici **−0,273** :
une pente réelle, mais légère.

Et encore, sur un échantillon filtré. Sur le fichier entier, c'est autre
chose — à vous de le mesurer.

---

### ✏️ Exercice — Afficher la corrélation dans le titre

> **Votre mission :**
> - Tracer un **nuage de points** avec `qte` en abscisse et `prix` en ordonnée, sur le fichier entier cette fois.
> - `alpha=0.2` rend les points translucides : sans lui, 45 000 points forment une tache noire.
> - La corrélation est déjà calculée → `lien`. **Faites-la entrer dans le titre** avec un `f-string` : un lecteur doit voir le chiffre sans avoir à le demander.
> - *Rappel :* dans un `f-string`, `{nom}` est remplacé par la valeur de la variable.

In [ ]:
lien = round(complet["qte"].corr(complet["prix"]), 3)   ## deja ecrit
titre = f"Quantite et prix unitaire (correlation {____})"

complet.plot(kind="____", x="qte", y="prix", alpha=0.2, figsize=(7, 4))
plt.title(titre)
plt.show()

print(titre)

In [ ]:
verifier("correlation calculee", lien == -0.024,
         "corr() entre les deux colonnes, arrondi a 3 decimales")
verifier("correlation dans le titre",
         titre == "Quantite et prix unitaire (correlation -0.024)",
         "dans un f-string, {lien} est remplace par la valeur de lien")

---

### ✏️ Exercice — Choisir le bon graphique

> **Votre mission :**
> - À chaque question sa figure. Compléter la liste `reponses` avec les quatre types, **dans l'ordre des questions** :
> - 1. Comment le chiffre d'affaires évolue-t-il dans le temps ?
> - 2. Quel pays est le plus gros marché ?
> - 3. Comment les prix sont-ils répartis ?
> - 4. Les grosses quantités vont-elles avec les prix bas ?

In [ ]:
reponses = ["____", "____", "____", "____"]

print(reponses)

In [ ]:
verifier("le bon graphique pour la bonne question",
         reponses == ["line", "barh", "hist", "scatter"],
         "une evolution, un classement, une repartition, une relation")

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| total par groupe | `df.groupby("pays")["ca"].sum()` |
| plusieurs indicateurs | `df.groupby("pays").agg(ca=("ca", "sum"), n=("cmd_id", "nunique"))` |
| classer un résultat | `serie.sort_values(ascending=False)` |
| les 5 plus grands | `serie.nlargest(5)`, ou `df.nlargest(5, "ca")` |
| joindre deux tables | `a.merge(b, on="client_id")` |
| croiser deux dimensions (en comptant) | `pd.crosstab(df["pays"], df["segment"])` |
| croiser en additionnant une colonne | le même, plus `values=df["ca"], aggfunc="sum"` |
| grouper sur deux clés | `df.groupby(["mois", "pays"])["ca"].sum()` |
| passer la seconde clé en colonnes | `serie.unstack()` |
| mesurer un lien entre deux colonnes | `df["qte"].corr(df["prix"])` — entre −1 et +1 |

| Votre question | Le graphique | La commande |
|---|---|---|
| comment ça évolue ? | courbe | `serie.plot(kind="line")` |
| qui est le plus gros ? | barres | `serie.plot(kind="barh")` |
| comment c'est réparti ? | histogramme | `df["prix"].plot(kind="hist", bins=30)` |
| y a-t-il un lien ? | nuage de points | `df.plot(kind="scatter", x="qte", y="prix")` |

Et toujours :

```python
plt.title("Ce que montre le graphique")
plt.xlabel("Nom de l'axe (unite)")
plt.ylabel("Nom de l'axe (unite)")
plt.show()
```

## Les deux erreurs à ne jamais commettre

1. **Faire un `merge` sans vérifier le nombre de lignes avant et après.**
   Un `merge` peut silencieusement dupliquer ou faire disparaître des lignes.
   `print(len(a), "->", len(fusion))` : une seconde, et vous dormez tranquille.

2. **Confondre `count` et `nunique`.** `count` compte les lignes,
   `nunique` compte les valeurs distinctes. Une commande de 30 articles,
   c'est 30 lignes mais **une** commande.

## Les deux réflexes du graphique

1. **Un graphique sans titre ni unité n'est pas un livrable.** Si votre
   lecteur doit vous demander « c'est en quoi ? », vous avez raté.

2. **Regardez toujours ce que le graphique ne montre pas.** Un mois incomplet,
   une catégorie absente, un axe qui ne part pas de zéro.